In [0]:
USE CATALOG maven_catalog;
USE SCHEMA gold_schema;

1) KPI Cards

In [0]:
CREATE OR REFRESH LIVE TABLE revenue_trend_dashboard
COMMENT "Sorted monthly revenue trends for time-series dashboarding"
TBLPROPERTIES ("quality" = "gold")
AS
SELECT 
  year, 
  month, 
  month_name, 
  revenue
FROM LIVE.v_revenue_trend_monthly
ORDER BY year, month;

3) Region bar chart

In [0]:
CREATE OR REFRESH LIVE TABLE revenue_by_region_dashboard
COMMENT "Regional revenue leaderboard for map and bar chart visualizations"
TBLPROPERTIES ("quality" = "gold")
AS
SELECT 
  sales_region, 
  revenue
FROM LIVE.v_revenue_by_region
ORDER BY revenue DESC;

4) Top products (donut or bar)

In [0]:
CREATE OR REFRESH LIVE TABLE top_10_products_dashboard
COMMENT "Top 10 products by revenue for the 'Best Sellers' dashboard component"
TBLPROPERTIES ("quality" = "gold")
AS
SELECT 
  product_name, 
  revenue
FROM LIVE.v_top_products
ORDER BY revenue DESC
LIMIT 10;

5) YoY growth (if you have multiple years)

In [0]:
CREATE LIVE VIEW v_yoy_growth
COMMENT "YoY revenue growth by year based on fact_sales and dim_calendar"
AS
WITH yr AS (
  SELECT
    dc.year,
    SUM(fs.total_revenue) AS revenue
  FROM LIVE.fact_sales fs
  JOIN LIVE.dim_calendar dc
    ON fs.transaction_date = dc.date_key
  GROUP BY dc.year
)
SELECT
  year,
  revenue,
  (revenue - LAG(revenue) OVER (ORDER BY year))
    / NULLIF(LAG(revenue) OVER (ORDER BY year), 0) AS yoy_growth
FROM yr
ORDER BY year;

